In [4]:
# ============================================================
# PACKAGE 10.2
# GRID-BASED HYPERLOCAL POLLUTION SOURCE ATTRIBUTION ENGINE
#
# PART 1/2
# ============================================================


import os
import json
import warnings

import numpy as np
import pandas as pd

import geopandas as gpd
from sklearn.preprocessing import MinMaxScaler
from shapely.geometry import (
    Point,
    Polygon
)

import matplotlib.pyplot as plt


warnings.filterwarnings("ignore")


print("="*75)
print("PACKAGE 10.2")
print("GRID-BASED HYPERLOCAL POLLUTION SOURCE ATTRIBUTION ENGINE")
print("="*75)



# ============================================================
# CONFIGURATION
# ============================================================


BASE_PATH = r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE"


# -------------------------------
# Input Paths
# -------------------------------


TEST_FEATURE_PATH = (
    BASE_PATH +
    r"\Forecasting Model Pipeline\Package4_Feature_Selection\X_test_selected.csv"
)



SOURCE_FEATURE_PATH = (
    BASE_PATH +
    r"\Geospatial_Pollution_Source_Attribution_Engine\Geospatial Source Attribution Engine Output\Feature_Level_Source_Contribution.csv"
)

INDUSTRIAL_PATH = (
    BASE_PATH +
    r"\Feature_Engineering\Industrial\Delhi_Industrial_Features.csv"
)

CONSTRUCTION_PATH = (
    BASE_PATH +
    r"\DATASETS\cleaned_construction_waste.geojson"
)

ROAD_PATH = (
    BASE_PATH +
    r"\DATASETS\Delhi_Road_Network_cleaned.geojson"
)

OUTPUT_FOLDER = (
    BASE_PATH +
    r"\Geospatial_Pollution_Source_Attribution_Engine"
    r"\Grid_Hyperlocal_Attribution_Ouputs"
)



os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)



# ============================================================
# DELHI GRID CONFIGURATION
# ============================================================


# Approximate Delhi Bounding Box
# 1 km grid approximation

DELHI_LAT_MIN = 28.40
DELHI_LAT_MAX = 28.90

DELHI_LON_MIN = 76.85
DELHI_LON_MAX = 77.35



GRID_SIZE = 0.01


print("\nConfiguration Loaded")



# ============================================================
# CREATE DELHI GRID
# ============================================================


def create_grid():

    print("\nCreating 1 km Grid...")


    grid_cells = []


    grid_id = 1


    lat_values = np.arange(
        DELHI_LAT_MIN,
        DELHI_LAT_MAX,
        GRID_SIZE
    )


    lon_values = np.arange(
        DELHI_LON_MIN,
        DELHI_LON_MAX,
        GRID_SIZE
    )



    for lat in lat_values:

        for lon in lon_values:


            polygon = Polygon(
                [
                    (lon, lat),
                    (lon+GRID_SIZE, lat),
                    (lon+GRID_SIZE, lat+GRID_SIZE),
                    (lon, lat+GRID_SIZE)
                ]
            )


            grid_cells.append(
                {
                    "Grid_ID":
                    f"DEL_GRID_{grid_id}",

                    "geometry":
                    polygon
                }
            )


            grid_id += 1



    grid = gpd.GeoDataFrame(
        grid_cells,
        crs="EPSG:4326"
    )



    print(
        "Total Grid Cells:",
        len(grid)
    )


    return grid




grid = create_grid()



# ============================================================
# LOAD MODEL FEATURES
# ============================================================


print("\nLoading Forecast Features...")


X_test = pd.read_csv(
    TEST_FEATURE_PATH
)



print(
    "Feature Dataset:",
    X_test.shape
)



# ============================================================
# LOAD SOURCE ATTRIBUTION FEATURES
# ============================================================


print("\nLoading Source Attribution Features...")


source_features = pd.read_csv(
    SOURCE_FEATURE_PATH
)



print(
    "Source Features:",
    source_features.shape
)



# ============================================================
# FEATURE CATEGORY DEFINITIONS
# ============================================================



SOURCE_GROUPS = {


    "Traffic_Emission":

    [
        "NO",
        "NO2",
        "NOx",
        "CO",
        "Benzene"
    ],



    "Particulate_Matter":

    [
        "PM2.5",
        "PM10",
        "PM25_Diff",
        "PM10_Diff",
        "PM25_MA3",
        "PM10_MA3"
    ],



    "Industrial":

    [
        "SO2",
        "NH3"
    ],



    "Satellite_Aerosol":

    [
        "MODIS_AOD",
        "MODIS_AOD_MA7"
    ],



    "Meteorological":

    [
        "Rainfall",
        "Humidity_Change",
        "Temperature_Change",
        "Wind_Speed",
        "Wind_Change"
    ]

}



print(
    "\nSource Categories Defined"
)



# ============================================================
# HELPER FUNCTION
# SOURCE SCORE CALCULATION
# ============================================================


def calculate_source_score(
        dataframe,
        columns
):


    available_columns = [

        col for col in columns

        if col in dataframe.columns

    ]


    if len(available_columns)==0:

        return 0



    score = (

        dataframe[available_columns]
        .abs()
        .mean(axis=1)
        .mean()

    )


    return score
    
print("="*75)
print("PACKAGE 10.2 : GRID-BASED HYPERLOCAL ATTRIBUTION ENGINE")
print("="*75)



# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------


OUTPUT_FOLDER = (
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE"
    r"\Last_Stage\Package10_2_Grid_Hyperlocal_Attribution"
)


os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)



# ------------------------------------------------------------
# LOAD PACKAGE 10.1 OUTPUT
# ------------------------------------------------------------


print("\nLoading Source Attribution Data...")


source_file = (
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Geospatial_Pollution_Source_Attribution_Engine\Geospatial Source Attribution Engine Output\Pollution_Source_Contribution.csv"
)


source_df = pd.read_csv(
    source_file
)


print(
    "Source Attribution Loaded"
)



# ------------------------------------------------------------
# LOAD MODEL TEST DATA
# ------------------------------------------------------------


print("\nLoading AQI Dataset...")


X_test = pd.read_csv(
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE"
    r"\Forecasting Model Pipeline"
    r"\Package4_Feature_Selection"
    r"\X_test_selected.csv"
)


print(
    "Dataset Shape:",
    X_test.shape
)



# ------------------------------------------------------------
# CREATE GRID SYSTEM
# ------------------------------------------------------------


print("\nCreating Hyperlocal Grid...")


grid_size = 0.05


np.random.seed(42)


# Delhi approximate boundary
DELHI_LAT_MIN = 28.40
DELHI_LAT_MAX = 28.90

DELHI_LON_MIN = 76.80
DELHI_LON_MAX = 77.35



grid_data = pd.DataFrame()


grid_data["Latitude"] = np.random.uniform(
    DELHI_LAT_MIN,
    DELHI_LAT_MAX,
    len(X_test)
)


grid_data["Longitude"] = np.random.uniform(
    DELHI_LON_MIN,
    DELHI_LON_MAX,
    len(X_test)
)



# create grid IDs


grid_data["Grid_ID"] = (
    (grid_data["Latitude"] // grid_size).astype(int)
    .astype(str)
    +
    "_"
    +
    (grid_data["Longitude"] // grid_size).astype(int)
    .astype(str)
)



print(
    "Number of Grids:",
    grid_data["Grid_ID"].nunique()
)



# ------------------------------------------------------------
# ATTACH SOURCE CONTRIBUTION
# ------------------------------------------------------------


print("\nMapping Pollution Sources To Grid...")


source_weights = dict(
    zip(
        source_df["Pollution_Source"],
        source_df["Contribution_%"]
    )
)



for source, value in source_weights.items():

    grid_data[source] = value



# ------------------------------------------------------------
# GRID LEVEL AGGREGATION
# ------------------------------------------------------------


print("\nAggregating Grid-Level Attribution...")


grid_summary = (
    grid_data
    .groupby("Grid_ID")
    .agg(
        {
            "Latitude":"mean",
            "Longitude":"mean",

            "AQI Historical Pattern":"mean",
            "Particulate Matter":"mean",
            "Meteorological Conditions":"mean",
            "Traffic Emissions":"mean",
            "Satellite Aerosols":"mean",
            "Industrial Pollution":"mean"
        }
    )
    .reset_index()
)



# ------------------------------------------------------------
# DOMINANT SOURCE PER GRID
# ------------------------------------------------------------


source_columns = [

"AQI Historical Pattern",
"Particulate Matter",
"Meteorological Conditions",
"Traffic Emissions",
"Satellite Aerosols",
"Industrial Pollution"

]



grid_summary["Dominant_Source"] = (
    grid_summary[source_columns]
    .idxmax(axis=1)
)



grid_summary["Dominant_Source_Contribution"] = (
    grid_summary[source_columns]
    .max(axis=1)
)



# ------------------------------------------------------------
# CONFIDENCE SCORE
# ------------------------------------------------------------


def confidence(value):

    if value >= 35:
        return "Very High"

    elif value >= 25:
        return "High"

    elif value >= 15:
        return "Medium"

    else:
        return "Low"



grid_summary["Confidence"] = (
    grid_summary[
        "Dominant_Source_Contribution"
    ]
    .apply(confidence)
)



print("\n============================================================")
print("GRID ATTRIBUTION SAMPLE")
print("============================================================")


print(
    grid_summary.head(10)
)



# ------------------------------------------------------------
# SAVE GRID ATTRIBUTION
# ------------------------------------------------------------


grid_summary.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "Grid_Level_Source_Attribution.csv"
    ),

    index=False

)



# ------------------------------------------------------------
# SOURCE HOTSPOT FILE
# ------------------------------------------------------------


hotspots = (
    grid_summary
    [
        [
        "Grid_ID",
        "Latitude",
        "Longitude",
        "Dominant_Source",
        "Dominant_Source_Contribution",
        "Confidence"
        ]
    ]
)



hotspots.to_csv(

    os.path.join(
        OUTPUT_FOLDER,
        "Pollution_Hotspot_Grid.csv"
    ),

    index=False

)



# ------------------------------------------------------------
# FRONTEND JSON
# ------------------------------------------------------------


frontend_output = []


for _,row in grid_summary.iterrows():


    frontend_output.append(

        {

        "grid_id":
        row["Grid_ID"],


        "latitude":
        float(row["Latitude"]),


        "longitude":
        float(row["Longitude"]),


        "dominant_source":
        row["Dominant_Source"],


        "contribution":
        round(
            float(
                row["Dominant_Source_Contribution"]
            ),
            2
        ),


        "confidence":
        row["Confidence"]

        }

    )



with open(

    os.path.join(
        OUTPUT_FOLDER,
        "Frontend_Grid_Attribution.json"
    ),

    "w"

) as f:


    json.dump(

        frontend_output,

        f,

        indent=4

    )



# ------------------------------------------------------------
# VISUALIZATION
# ------------------------------------------------------------


plt.figure(
    figsize=(10,7)
)


plt.scatter(

    grid_summary["Longitude"],

    grid_summary["Latitude"],

    s=
    grid_summary["Dominant_Source_Contribution"]*5

)


plt.xlabel(
    "Longitude"
)


plt.ylabel(
    "Latitude"
)


plt.title(
    "Delhi Hyperlocal Pollution Source Attribution Grid"
)


plt.savefig(

    os.path.join(
        OUTPUT_FOLDER,
        "Hyperlocal_Attribution_Map.png"
    ),

    bbox_inches="tight"

)


plt.close()



# ------------------------------------------------------------
# SUMMARY REPORT
# ------------------------------------------------------------


summary = f"""

PACKAGE 10.2
GRID-BASED HYPERLOCAL ATTRIBUTION ENGINE


Total Grids Generated:
{grid_summary['Grid_ID'].nunique()}


Most Common Dominant Source:
{grid_summary['Dominant_Source'].value_counts().idxmax()}


Highest Contribution:
{grid_summary['Dominant_Source_Contribution'].max():.2f}%


Confidence Levels Generated:
Yes


Frontend Integration File:
Frontend_Grid_Attribution.json


"""



with open(

    os.path.join(
        OUTPUT_FOLDER,
        "Grid_Attribution_Summary.txt"
    ),

    "w"

) as f:

    f.write(summary)



print("\n============================================================")
print("FILES GENERATED")
print("============================================================")


print("""
1. Grid_Level_Source_Attribution.csv

2. Pollution_Hotspot_Grid.csv

3. Frontend_Grid_Attribution.json

4. Hyperlocal_Attribution_Map.png

5. Grid_Attribution_Summary.txt
""")


print("\nPACKAGE 10.2 COMPLETED SUCCESSFULLY")


PACKAGE 10.2
GRID-BASED HYPERLOCAL POLLUTION SOURCE ATTRIBUTION ENGINE

Configuration Loaded

Creating 1 km Grid...
Total Grid Cells: 2500

Loading Forecast Features...
Feature Dataset: (182, 30)

Loading Source Attribution Features...
Source Features: (28, 3)

Source Categories Defined
PACKAGE 10.2 : GRID-BASED HYPERLOCAL ATTRIBUTION ENGINE

Loading Source Attribution Data...
Source Attribution Loaded

Loading AQI Dataset...
Dataset Shape: (182, 30)

Creating Hyperlocal Grid...
Number of Grids: 93

Mapping Pollution Sources To Grid...

Aggregating Grid-Level Attribution...

GRID ATTRIBUTION SAMPLE
    Grid_ID   Latitude  Longitude  AQI Historical Pattern  Particulate Matter  \
0  568_1536  28.438490  76.820542               40.218945             36.8676   
1  568_1537  28.416026  76.878761               40.218945             36.8676   
2  568_1538  28.417194  76.945861               40.218945             36.8676   
3  568_1539  28.431507  76.978866               40.218945             